In [3]:
from pathlib import Path
import polars as pl

DATA_DIR = Path("data/raw")

for file in DATA_DIR.glob("*.csv"):
    df = pl.read_csv(file, try_parse_dates=True)

    print(f"\n===== {file.name} =====")
    print("Rows:", df.height)
    print("Columns:", df.width)

    print("\nSchema:")
    print(df.schema)

    print("\nColumn profiling:")

    for col in df.columns:
        null_count = df[col].null_count()
        cardinality = df[col].n_unique()

        print({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": null_count,
            "null_pct": round(null_count / df.height * 100, 2),
            "cardinality": cardinality,
            "unique_pct": round(cardinality / df.height * 100, 2)
        })

    duplicate_rows = df.height - df.unique().height
    print("\nDuplicate rows:", duplicate_rows)

    timestamp_cols = [
        c for c, dtype in df.schema.items()
        if dtype in (pl.Date, pl.Datetime)
    ]

    for col in timestamp_cols:
        print(
            col,
            df.select(
                pl.col(col).min().alias("min"),
                pl.col(col).max().alias("max")
            )
        )
    


===== customers.csv =====
Rows: 20000
Columns: 7

Schema:
Schema({'customer_id': Int64, 'name': String, 'email': String, 'country': String, 'age': Int64, 'signup_date': Date, 'marketing_opt_in': Boolean})

Column profiling:
{'column': 'customer_id', 'dtype': 'Int64', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 20000, 'unique_pct': 100.0}
{'column': 'name', 'dtype': 'String', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 17915, 'unique_pct': 89.58}
{'column': 'email', 'dtype': 'String', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 20000, 'unique_pct': 100.0}
{'column': 'country', 'dtype': 'String', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 17, 'unique_pct': 0.08}
{'column': 'age', 'dtype': 'Int64', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 58, 'unique_pct': 0.29}
{'column': 'signup_date', 'dtype': 'Date', 'null_count': 0, 'null_pct': 0.0, 'cardinality': 2131, 'unique_pct': 10.66}
{'column': 'marketing_opt_in', 'dtype': 'Boolean', 'null_count': 0, 'null_pct': 0

Relationship

In [4]:
customers = pl.read_csv("data/raw/customers.csv")
sessions = pl.read_csv("data/raw/sessions.csv")

orphan_sessions = sessions.join(
    customers.select("customer_id"),
    on="customer_id",
    how="anti"
)

print("Invalid customer FK:", orphan_sessions.height)

Invalid customer FK: 0


Event sequences

In [5]:
events = pl.read_csv("data/raw/events.csv")

events = events.sort(
    ["session_id", "timestamp"]
)

sequences = (
    events
    .group_by("session_id")
    .agg(
        pl.col("event_type").alias("event_sequence")
    )
)

print(sequences.head())

shape: (5, 2)
┌────────────┬─────────────────────────────────┐
│ session_id ┆ event_sequence                  │
│ ---        ┆ ---                             │
│ i64        ┆ list[str]                       │
╞════════════╪═════════════════════════════════╡
│ 1          ┆ ["page_view", "page_view", … "… │
│ 2          ┆ ["page_view", "page_view", … "… │
│ 3          ┆ ["page_view", "add_to_cart", …… │
│ 4          ┆ ["page_view", "page_view"]      │
│ 5          ┆ ["page_view", "page_view", … "… │
└────────────┴─────────────────────────────────┘


Customer → session → order relationships

In [6]:
orders = pl.read_csv("data/raw/orders.csv")
customer_sessions = (
    sessions
    .group_by("customer_id")
    .agg(
        pl.len().alias("session_count")
    )
)


In [7]:
print(
    customer_sessions
    .select(
        pl.col("session_count").min().alias("min"),
        pl.col("session_count").mean().alias("mean"),
        pl.col("session_count").median().alias("median"),
        pl.col("session_count").max().alias("max")
    )
)

shape: (1, 4)
┌─────┬──────────┬────────┬─────┐
│ min ┆ mean     ┆ median ┆ max │
│ --- ┆ ---      ┆ ---    ┆ --- │
│ u32 ┆ f64      ┆ f64    ┆ u32 │
╞═════╪══════════╪════════╪═════╡
│ 1   ┆ 6.016546 ┆ 6.0    ┆ 17  │
└─────┴──────────┴────────┴─────┘


## Relationship & Business Integrity Tests

These checks validate the eight critical relationships in the source and distinguish true data defects from structural nulls or undocumented grain.

### Key assumptions

- Source IDs are integer business keys. `events.product_id` is cast from `Float64` to `Int64` because CSV inference promotes the column when nulls are present.
- A purchase-to-order match is deterministic only when customer, timestamp, amount, discount, and payment method all agree.
- Financial comparisons use USD-cent precision. A one-cent difference after percentage discounting is treated as expected rounding.


In [8]:
# Load all source tables once for integrity tests
customers = pl.read_csv(DATA_DIR / "customers.csv", try_parse_dates=True)
sessions = pl.read_csv(DATA_DIR / "sessions.csv", try_parse_dates=True)
events = pl.read_csv(DATA_DIR / "events.csv", try_parse_dates=True)
orders = pl.read_csv(DATA_DIR / "orders.csv", try_parse_dates=True)
order_items = pl.read_csv(DATA_DIR / "order_items.csv", try_parse_dates=True)
products = pl.read_csv(DATA_DIR / "products.csv", try_parse_dates=True)
reviews = pl.read_csv(DATA_DIR / "reviews.csv", try_parse_dates=True)


### 1. Referential integrity across source tables

In [9]:
def fk_result(relationship, child, child_keys, parent, parent_keys):
    # Normalize integer-valued IDs before joining (notably events.product_id).
    left = child.with_columns([pl.col(c).cast(pl.Int64).alias(c) for c in child_keys])
    right = (
        parent.select(parent_keys)
        .rename(dict(zip(parent_keys, child_keys)))
        .with_columns([pl.col(c).cast(pl.Int64).alias(c) for c in child_keys])
        .unique()
    )
    orphan = left.join(right, on=child_keys, how="anti")
    return {
        "relationship": relationship,
        "child_rows": child.height,
        "orphan_rows": orphan.height,
        "orphan_keys": orphan.select(child_keys).unique().height,
        "match_rate_pct": round((child.height - orphan.height) / child.height * 100, 4),
    }

product_events = events.filter(pl.col("product_id").is_not_null())
fk_results = pl.DataFrame([
    fk_result("customers -> sessions", sessions, ["customer_id"], customers, ["customer_id"]),
    fk_result("sessions -> events", events, ["session_id"], sessions, ["session_id"]),
    fk_result("customers -> orders", orders, ["customer_id"], customers, ["customer_id"]),
    fk_result("orders -> order_items", order_items, ["order_id"], orders, ["order_id"]),
    fk_result("products -> events", product_events, ["product_id"], products, ["product_id"]),
    fk_result("products -> order_items", order_items, ["product_id"], products, ["product_id"]),
    fk_result("products -> reviews", reviews, ["product_id"], products, ["product_id"]),
    fk_result("orders -> reviews", reviews, ["order_id"], orders, ["order_id"]),
    fk_result("orders + products -> reviews", reviews, ["order_id", "product_id"], order_items, ["order_id", "product_id"]),
])
fk_results


relationship,child_rows,orphan_rows,orphan_keys,match_rate_pct
str,i64,i64,i64,f64
"""customers -> sessions""",120000,0,0,100.0
"""sessions -> events""",760958,0,0,100.0
"""customers -> orders""",33580,0,0,100.0
"""orders -> order_items""",59163,0,0,100.0
"""products -> events""",682469,0,0,100.0
"""products -> order_items""",59163,0,0,100.0
"""products -> reviews""",10780,0,0,100.0
"""orders -> reviews""",10780,0,0,100.0
"""orders + products -> reviews""",10780,0,0,100.0


**Interpretation:** All tested foreign keys have 100% coverage. The apparent extra product cardinality in `events` was a profiling artifact: Polars counted `null` as one distinct value. After excluding structural nulls and normalizing the numeric type, there are no orphan product IDs.

### 2. Purchase events mapped to orders

In [10]:
purchases = (
    events.filter(pl.col("event_type") == "purchase")
    .join(sessions.select("session_id", "customer_id"), on="session_id", how="left")
    .with_columns(
        pl.col("timestamp").cast(pl.Datetime("us")).alias("match_time"),
        pl.col("amount_usd").round(2).alias("match_amount"),
        pl.col("discount_pct").cast(pl.Float64).round(6).alias("match_discount"),
        pl.col("payment").str.to_lowercase().alias("match_payment"),
    )
)

orders_for_match = orders.with_columns(
    pl.col("order_time").cast(pl.Datetime("us")).alias("match_time"),
    pl.col("total_usd").round(2).alias("match_amount"),
    pl.col("discount_pct").cast(pl.Float64).round(6).alias("match_discount"),
    pl.col("payment_method").str.to_lowercase().alias("match_payment"),
)

purchase_order_key = [
    "customer_id", "match_time", "match_amount", "match_discount", "match_payment"
]
order_candidates = orders_for_match.group_by(purchase_order_key).agg(
    pl.len().alias("order_candidates"),
    pl.col("order_id"),
)
purchase_matches = purchases.join(order_candidates, on=purchase_order_key, how="left")

purchase_match_summary = purchase_matches.select(
    pl.len().alias("purchase_events"),
    pl.col("order_candidates").is_null().sum().alias("unmatched"),
    (pl.col("order_candidates") == 1).sum().alias("unique_matches"),
    (pl.col("order_candidates") > 1).sum().alias("ambiguous_matches"),
).with_columns(
    (pl.col("unique_matches") / pl.col("purchase_events") * 100).round(4).alias("unique_match_rate_pct")
)
purchase_match_summary


purchase_events,unmatched,unique_matches,ambiguous_matches,unique_match_rate_pct
u32,u32,u32,u32,f64
33580,0,33580,0,100.0


**Interpretation:** All 33,580 purchase events map uniquely to exactly one order; there are no unmatched or ambiguous cases. The relationship is reconstructable, but the source should ideally expose `order_id` on purchase events instead of relying on a multi-column inferred key.

### 3. `order_items` grain and duplicate diagnosis

In [11]:
full_duplicate_groups = order_items.group_by(order_items.columns).len().filter(pl.col("len") > 1)
order_product_groups = order_items.group_by(["order_id", "product_id"]).agg(
    pl.len().alias("rows"),
    pl.col("quantity").sum().alias("quantity_sum"),
    pl.col("line_total_usd").sum().round(2).alias("line_total_sum"),
    pl.col("unit_price_usd").n_unique().alias("distinct_unit_prices"),
).filter(pl.col("rows") > 1)

duplicate_summary = pl.DataFrame({
    "metric": [
        "exact duplicate groups", "extra exact duplicate rows",
        "duplicate (order_id, product_id) groups", "extra order-product rows"
    ],
    "value": [
        full_duplicate_groups.height,
        full_duplicate_groups.select((pl.col("len") - 1).sum()).item(),
        order_product_groups.height,
        order_product_groups.select((pl.col("rows") - 1).sum()).item(),
    ],
})
duplicate_summary


metric,value
str,i64
"""exact duplicate groups""",73
"""extra exact duplicate rows""",73
"""duplicate (order_id, product_i…",110
"""extra order-product rows""",110


In [12]:
# Check line arithmetic and whether raw vs deduplicated lines reconcile to order subtotal.
line_math = order_items.with_columns(
    (pl.col("unit_price_usd") * pl.col("quantity") - pl.col("line_total_usd")).abs().alias("difference")
)
raw_item_totals = order_items.group_by("order_id").agg(
    pl.col("line_total_usd").sum().round(2).alias("raw_items_subtotal")
)
dedup_item_totals = order_items.unique().group_by("order_id").agg(
    pl.col("line_total_usd").sum().round(2).alias("dedup_items_subtotal")
)
duplicate_reconciliation = (
    orders.select("order_id", "subtotal_usd")
    .join(raw_item_totals, on="order_id", how="left")
    .join(dedup_item_totals, on="order_id", how="left")
    .select(
        (pl.col("subtotal_usd") - pl.col("raw_items_subtotal")).abs().gt(0.01).sum().alias("raw_subtotal_failures"),
        (pl.col("subtotal_usd") - pl.col("dedup_items_subtotal")).abs().gt(0.01).sum().alias("failures_after_exact_dedup"),
        (pl.col("subtotal_usd") - pl.col("dedup_items_subtotal")).abs().max().alias("max_error_after_dedup_usd"),
    )
)
pl.concat([
    line_math.select(
        pl.lit("line_total = unit_price * quantity").alias("check"),
        (pl.col("difference") > 0.01).sum().alias("failures"),
        pl.col("difference").max().alias("max_difference_usd"),
    ),
    duplicate_reconciliation.select(
        pl.lit("raw item sum = order subtotal").alias("check"),
        pl.col("raw_subtotal_failures").alias("failures"),
        pl.lit(0.0).alias("max_difference_usd"),
    ),
])


check,failures,max_difference_usd
str,u32,f64
"""line_total = unit_price * quan…",0,2.2737e-13
"""raw item sum = order subtotal""",0,0.0


In [13]:
duplicate_reconciliation


raw_subtotal_failures,failures_after_exact_dedup,max_error_after_dedup_usd
u32,u32,f64
0,73,303.3


**Interpretation:** `(order_id, product_id)` is not the source grain: 110 order-product pairs contain multiple rows, including 73 exact duplicate pairs. These rows must **not** be dropped, because the raw rows reconcile every order subtotal while exact deduplication breaks 73 orders by as much as $303.30. The defensible grain is **one row per unkeyed order line**. The source has omitted an `order_item_id`/line number, so identical legitimate lines cannot be distinguished. Add a surrogate row key during ingestion, but preserve every raw row.

### 4. Financial reconciliation

In [14]:
financial_reconciliation = (
    orders.join(raw_item_totals, on="order_id", how="left")
    .with_columns(
        (pl.col("subtotal_usd") - pl.col("raw_items_subtotal")).abs().alias("subtotal_difference"),
        (pl.col("subtotal_usd") * (1 - pl.col("discount_pct") / 100)).round(2).alias("expected_total"),
    )
    .with_columns(
        (pl.col("total_usd") - pl.col("expected_total")).abs().alias("total_difference")
    )
)

financial_summary = financial_reconciliation.select(
    pl.len().alias("orders"),
    pl.col("raw_items_subtotal").is_null().sum().alias("orders_without_items"),
    (pl.col("subtotal_difference") > 0.01).sum().alias("subtotal_failures_gt_1_cent"),
    pl.col("subtotal_difference").max().alias("max_subtotal_difference"),
    (pl.col("total_difference") > 0.011).sum().alias("discount_total_failures_gt_1_cent"),
    pl.col("total_difference").max().alias("max_total_difference"),
)
financial_summary


orders,orders_without_items,subtotal_failures_gt_1_cent,max_subtotal_difference,discount_total_failures_gt_1_cent,max_total_difference
u32,u32,u32,f64,u32,f64
33580,0,0,0.0,0,0.01


**Interpretation:** Every order has line items; line sums reconcile exactly to `subtotal_usd`. `total_usd` follows `round(subtotal_usd × (1 − discount_pct / 100), 2)`. The maximum apparent difference is one cent and is attributable to decimal/float rounding, not a financial integrity failure.

## Data-quality conclusions

| Question | Result | Decision |
|---|---|---|
| Purchase event maps 1:1 to order? | Yes: 33,580/33,580 unique matches | Reconstruct with the documented composite key; flag missing native `order_id` as a source-model limitation |
| Are 73 duplicate `order_items` errors? | No evidence of erroneous duplication; removing them breaks 73 reconciliations | Preserve all rows and model grain as unkeyed order line; generate a surrogate line key |
| Are there orphan product IDs in events? | No: 0/682,469 non-null product references are orphaned | Cast product IDs to integer before relationship testing; treat null as structural |
| Other source relationships | All tested relationships have 100% coverage | Encode stable FK checks as dbt relationship tests |
| Financial reconciliation | Passes at cent precision | Encode arithmetic and reconciliation as singular dbt tests |

**Overall assessment:** The source is internally consistent and suitable for the project. The main modeling weakness is not corrupt data, but the absence of explicit keys linking purchase events to orders and identifying individual order-item lines.
